In [3]:
import pandas as pd

df_audience = pd.read_excel('Данные для тестового задания.xlsx', sheet_name='Данные об аудитории')
df_ab = pd.read_excel('Данные для тестового задания.xlsx', sheet_name='Данные АБ тестов')
df_listers = pd.read_excel('Данные для тестового задания.xlsx', sheet_name='Листеры')

In [4]:
# Вопрос 1. Во вкладке "Данные об аудитории" информация о пользователях, посетивших наше приложение в ноябре. Чему равен MAU продукта? 

mau = df_audience['user_id'].nunique()
print(f"MAU продукта: {mau}")

MAU продукта: 7639


In [6]:
# Вопрос 2. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен DAU

dau_by_day = df_audience.groupby('date')['user_id'].nunique()
average_dau = dau_by_day.mean()
print(f"Средний DAU за месяц: {average_dau:.2f}")

Средний DAU за месяц: 560.47


In [7]:
# Вопрос 3. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен retention первого дня у пользователей, пришедших в продукт 1 ноября 

cohort_nov1 = df_audience[(df_audience['date'].dt.month == 11) & (df_audience['date'].dt.day == 1)]['user_id'].unique()
cohort_size = len(cohort_nov1)

active_nov2 = df_audience[(df_audience['date'].dt.month == 11) & (df_audience['date'].dt.day == 2)]['user_id'].unique()

returned_users = set(cohort_nov1).intersection(set(active_nov2))
returned_count = len(returned_users)

retention_day1 = (returned_count / cohort_size) * 100 if cohort_size > 0 else 0

print(f"Retention 1-го дня: {retention_day1:.2f}%")

Retention 1-го дня: 26.65%


In [9]:
# Вопрос 5. Во вкладке "Данные об аудитории" есть информация о том, сколько объявлений посмотрел каждый пользователь (view_adverts). Посчитайте пользовательскую конверсию в просмотр объявления за ноябрь? (в пользователях)

total_users = df_audience['user_id'].nunique()
converted_users = df_audience[df_audience['view_adverts'] > 0]['user_id'].nunique()
conversion_rate = (converted_users / total_users) * 100

print(f"Пользовательская конверсия: {conversion_rate:.2f}%")

Пользовательская конверсия: 46.31%


In [11]:
# Вопрос 6. Используя информацию из вкладки "Данные об аудитории", посчитайте среднее количество просмотренных объявлений на пользователя в ноябре

total_views = df_audience['view_adverts'].sum()
unique_users = df_audience['user_id'].nunique()
avg_per_unique_user = total_views / unique_users

print(f"Среднее на пользователя: {avg_per_unique_user:.2f}")

Среднее на пользователя: 2.87


In [12]:
# Вопрос 7.  Мы провели опрос среди 2000 пользователей. Из них 500 «критики», 1200 «сторонники» и 300 «нейтралы». Посчитайте, чему будет равен NPS 

promoters_pct = (1200 / 2000) * 100
detractors_pct = (500 / 2000) * 100

nps = promoters_pct - detractors_pct
print(f"NPS: {nps}%")

NPS: 35.0%


In [16]:
# Вопрос 8. Во вкладке "Данные АБ-тестов" результаты трех несвязанных АБ тестов для ARPU (общая выручка/общее количество пользователей). Посмотрите на результаты тестов и интерпретируйте их. Напишите значения p-value, которые вы получили. 

from scipy import stats
import numpy as np

df_ab['experiment_group'] = df_ab['experiment_group'].astype(str).str.strip()

experiments = sorted(df_ab['experiment_num'].unique())

for exp in experiments:
    exp_df = df_ab[df_ab['experiment_num'] == exp]
    
    groups = exp_df['experiment_group'].unique()
    
    if len(groups) < 2:
        print(f"В эксперименте №{exp} недостаточно групп для сравнения: {groups}")
        continue
        
    groups = sorted(groups)
    g1_name, g2_name = groups[0], groups[1]
    
    group_A = exp_df[exp_df['experiment_group'] == g1_name]['revenue']
    group_B = exp_df[exp_df['experiment_group'] == g2_name]['revenue']
    
    arpu_A = group_A.mean()
    arpu_B = group_B.mean()
    
    n_A = len(group_A)
    n_B = len(group_B)
    
    t_stat, p_value = stats.ttest_ind(group_A, group_B, equal_var=False)
    
    lift = ((arpu_B - arpu_A) / arpu_A) * 100 if arpu_A > 0 else 0
    
    print(f"Эксперимент №{exp}:")
    print(f"  Группа {g1_name} (Контроль): Users = {n_A}, ARPU = {arpu_A:.2f}")
    print(f"  Группа {g2_name} (Тест):    Users = {n_B}, ARPU = {arpu_B:.2f}")
    print(f"  Изменение ARPU:    {lift:+.2f}%")
    print(f"  p-value:            {p_value:.5f}")
    
    if p_value < 0.05:
        print("  Результат: СТАТИСТИЧЕСКИ ЗНАЧИМ! Различия между группами реальны.")
    else:
        print("  Результат: СТАТИСТИЧЕСКИ НЕЗНАЧИМ! Различия случайны, раскатывать нельзя.")
    print("-" * 50)

Эксперимент №1:
  Группа control (Контроль): Users = 465, ARPU = 722.46
  Группа test (Тест):    Users = 480, ARPU = 665.74
  Изменение ARPU:    -7.85%
  p-value:            0.68897
  Результат: СТАТИСТИЧЕСКИ НЕЗНАЧИМ! Различия случайны, раскатывать нельзя.
--------------------------------------------------
Эксперимент №2:
  Группа control (Контроль): Users = 465, ARPU = 704.65
  Группа test (Тест):    Users = 480, ARPU = 332.93
  Изменение ARPU:    -52.75%
  p-value:            0.00113
  Результат: СТАТИСТИЧЕСКИ ЗНАЧИМ! Различия между группами реальны.
--------------------------------------------------
Эксперимент №3:
  Группа control (Контроль): Users = 465, ARPU = 663.21
  Группа test (Тест):    Users = 480, ARPU = 998.67
  Изменение ARPU:    +50.58%
  p-value:            0.06032
  Результат: СТАТИСТИЧЕСКИ НЕЗНАЧИМ! Различия случайны, раскатывать нельзя.
--------------------------------------------------


In [18]:
# Вопрос 9. По датасету с листерами посчитайте средний доход на пользователя 

total_revenue = df_listers['revenue'].sum()
unique_listers = df_listers['user_id'].nunique()
arpu_unique = total_revenue / unique_listers

print(f"Средний доход на пользователя: {arpu_unique:.2f}")

Средний доход на пользователя: 156.48


In [20]:
# Вопрос 10. По датасету с листерами посчитайте медиану возраста пользователя 

median_unique = df_listers.groupby('user_id')['age'].first().median()

print(f"Медиана возраста: {median_unique}")


Медиана возраста: 28.0


In [21]:
# Вопрос 18. Были получены следующие результаты. Коллеги просят вас подтвердить их и сделать окончательный вывод по эксперименту. 
# Вариант A (контрольная группа) — 100 047 501 посетитель, 1003 платежа. 
# Вариант B (тестовая группа) — 100 001 055 посетителей, 1099 платежей. 
# Какие рекомендации вы бы дали, основываясь на этих данных? 

from statsmodels.stats.proportion import proportions_ztest

count = [1099, 1003]  # [Тест, Контроль]
nobs = [100001055, 100047501]

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print(f"p-value: {p_value:.5f}")

p-value: 0.03533
